In [1]:
%load_ext autoreload
%autoreload 2
%reload_ext autoreload

import nest_asyncio
nest_asyncio.apply()

import plotly.express as px
import plotly.graph_objects as go
import plotly.io as pio
pio.renderers.default = "vscode"            

import matplotlib.pyplot as plt
import matplotlib.pylab as pylab
import matplotlib.dates as mdates
plt.style.use('ggplot')
params = {'legend.fontsize': 'medium',
        'figure.figsize': (18, 8),
        'axes.labelsize': 'medium',
        'axes.titlesize':'medium',
        'xtick.labelsize':'medium',
        'ytick.labelsize':'medium'}
pylab.rcParams.update(params)

import pandas as pd
import numpy as np
import rateslib as rl
import QuantLib as ql

import time
import datetime
import pytz

NY_tz = pytz.timezone("America/New_York") 
CHI_tz = pytz.timezone("America/Chicago") 
LDN_tz = pytz.timezone("Europe/London") 
UTC_tz = pytz.timezone("UTC") 

import sys
sys.path.append("../../")

from RVUtils.plt_timeseries import make_secondary_axis_plot

In [3]:
from MDP.STIRConvexityAdjustment.STIRConvexityAdjustmentMDP import STIRConvexityAdjustmentMDP
from Query.IRSwaps.IRSwapQuery import IRSwapQuery, IRSwapStructure, IRSwapValue

mdp = STIRConvexityAdjustmentMDP(source_a="BARCHART_STIRF-RL", source_b="ERIS_EOD_LIVE-RL_BASIC-NOJUMPS")

eod = NY_tz.localize(datetime.datetime(2026, 3, 10, 17, 00))

q = IRSwapQuery(
    tenor="IMM_H29xIMM_Z29",
    value=IRSwapValue.CVX_ADJ_EMPIRICAL,
    market_request={
        "curve_name_a": "USD-SOFR-1D-Q16STIRT",
        "curve_name_b": "USD-SOFR-1D",
    },
)

request = q.build_mdp_request(eod)
pricer = mdp.get_pricer(request)
package, weights = q.resolve_package(pricer_or_curve=pricer)
val_map = q.build_value_map(pricer_or_curve=pricer, **dict(zip(("package", "risk_weights"), q.resolve_package(pricer_or_curve=pricer))))
cvx_adj_bps = float(val_map.apply(IRSwapValue.CVX_ADJ_EMPIRICAL))
print(cvx_adj_bps)

6.28942097885013
